In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold

data = pd.read_csv('dataset/training_data.csv', header=None, delimiter="\t", names=["labels", "text"])
data_test = pd.read_csv("dataset/testing_data.csv", header=None, delimiter="\t", names=["labels", "text"])

n_folds = 5

kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
data_test.head()

,labels,text
0,2,copycat muslim terrorist arrested with assault...
1,2,wow! chicago protester caught on camera admits...
2,2,germany's fdp look to fill schaeuble's big shoes
3,2,mi school sends welcome back packet warning ki...
4,2,u.n. seeks 'massive' aid boost amid rohingya '...


In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")


/opt/anaconda3/envs/nlp_env_2/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6149.31it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
data_encoded = model.encode(data.text.values)
data_test_encoded = model.encode(data_test.text.values)

In [8]:
# model - the model that will be fitted
# MODEL_NAME - with what name to save it 
# X - the data that is used in the KFold
# y - labels for the X
# test_data - the data that is used for LB 
def fit_predict_save(model, MODEL_NAME, X, y, test_data):
    d_e = X.copy()
    d_t_e = test_data.copy()
    oof = np.zeros((X.shape[0]))
    pred = np.zeros((test_data.shape[0]))

    for i, (train_index, test_index) in  enumerate(kf.split(d_e, y)):
        print("FOLD: ", i, "TRAIN:", train_index, "TEST:", test_index)
        X_train, X_test = d_e[train_index], d_e[test_index]
        y_train = y[train_index]

        model.fit(X_train, y_train)

        oof[test_index] = model.predict_proba(X_test)[:, 1] # for every index of test in every fold find the probability of 1, and as a result we will have all indexes covered
        pred += model.predict_proba(d_t_e)[:, 1] # find probability of 1 for every data_test item __5_times__ (n_folds)

    pred /= n_folds # since we summed 5 (n_folds) times, normalize it 

    # Save 
    pd.DataFrame({MODEL_NAME: pred}).to_csv(f"results/{MODEL_NAME}.csv", index=False)
    pd.DataFrame({MODEL_NAME + "_oof": oof}).to_csv(f"results/{MODEL_NAME}_oof.csv", index=False)

In [9]:
from sklearn.linear_model import LogisticRegression

fit_predict_save(LogisticRegression(), "LogisticRegression", data_encoded, data["labels"], data_test_encoded)


FOLD:  0 TRAIN: [    0     1     2 ... 34147 34149 34151] TEST: [    5     7    12 ... 34145 34148 34150]
FOLD:  1 TRAIN: [    0     1     2 ... 34148 34150 34151] TEST: [    9    15    25 ... 34144 34147 34149]
FOLD:  2 TRAIN: [    0     1     3 ... 34149 34150 34151] TEST: [    2     4    14 ... 34140 34142 34146]
FOLD:  3 TRAIN: [    0     1     2 ... 34148 34149 34150] TEST: [    3     6     8 ... 34133 34143 34151]
FOLD:  4 TRAIN: [    2     3     4 ... 34149 34150 34151] TEST: [    0     1    10 ... 34122 34124 34141]


In [47]:
from sklearn.naive_bayes import MultinomialNB
import numpy as np

# This model supports only non-negative values, so we need to shift the data to be non-negative
min_val = np.min([np.min(data_encoded), np.min(data_test_encoded)])
data_positive_encoded = data_encoded - min_val
data_test_positive_encoded = data_test_encoded - min_val

fit_predict_save(MultinomialNB(), "MultinomialNB", data_positive_encoded, data["labels"], data_test_positive_encoded)


FOLD:  0 TRAIN: [    0     1     2 ... 34147 34149 34151] TEST: [    5     7    12 ... 34145 34148 34150]
FOLD:  1 TRAIN: [    0     1     2 ... 34148 34150 34151] TEST: [    9    15    25 ... 34144 34147 34149]
FOLD:  2 TRAIN: [    0     1     3 ... 34149 34150 34151] TEST: [    2     4    14 ... 34140 34142 34146]
FOLD:  3 TRAIN: [    0     1     2 ... 34148 34149 34150] TEST: [    3     6     8 ... 34133 34143 34151]
FOLD:  4 TRAIN: [    2     3     4 ... 34149 34150 34151] TEST: [    0     1    10 ... 34122 34124 34141]


Much more on left bottom

In [48]:
from sklearn.ensemble import RandomForestClassifier

fit_predict_save(RandomForestClassifier(), "RandomForestClassifier", data_encoded, data["labels"], data_test_encoded)

FOLD:  0 TRAIN: [    0     1     2 ... 34147 34149 34151] TEST: [    5     7    12 ... 34145 34148 34150]
FOLD:  1 TRAIN: [    0     1     2 ... 34148 34150 34151] TEST: [    9    15    25 ... 34144 34147 34149]
FOLD:  2 TRAIN: [    0     1     3 ... 34149 34150 34151] TEST: [    2     4    14 ... 34140 34142 34146]
FOLD:  3 TRAIN: [    0     1     2 ... 34148 34149 34150] TEST: [    3     6     8 ... 34133 34143 34151]
FOLD:  4 TRAIN: [    2     3     4 ... 34149 34150 34151] TEST: [    0     1    10 ... 34122 34124 34141]


In [49]:
from sklearn.ensemble import GradientBoostingClassifier
 
fit_predict_save(GradientBoostingClassifier(), "GradientBoostingClassifier", data_encoded, data["labels"], data_test_encoded)

FOLD:  0 TRAIN: [    0     1     2 ... 34147 34149 34151] TEST: [    5     7    12 ... 34145 34148 34150]
FOLD:  1 TRAIN: [    0     1     2 ... 34148 34150 34151] TEST: [    9    15    25 ... 34144 34147 34149]
FOLD:  2 TRAIN: [    0     1     3 ... 34149 34150 34151] TEST: [    2     4    14 ... 34140 34142 34146]
FOLD:  3 TRAIN: [    0     1     2 ... 34148 34149 34150] TEST: [    3     6     8 ... 34133 34143 34151]
FOLD:  4 TRAIN: [    2     3     4 ... 34149 34150 34151] TEST: [    0     1    10 ... 34122 34124 34141]


In [52]:
from sklearn.ensemble import StackingClassifier

estimators = [
    ('lr', LogisticRegression()),
    ('nb', MultinomialNB()),
    ('rf', RandomForestClassifier())
]

fit_predict_save(StackingClassifier(estimators=estimators, final_estimator=LogisticRegression()), "StackingClassifier", data_positive_encoded, data["labels"], data_test_positive_encoded)

FOLD:  0 TRAIN: [    0     1     2 ... 34147 34149 34151] TEST: [    5     7    12 ... 34145 34148 34150]
FOLD:  1 TRAIN: [    0     1     2 ... 34148 34150 34151] TEST: [    9    15    25 ... 34144 34147 34149]
FOLD:  2 TRAIN: [    0     1     3 ... 34149 34150 34151] TEST: [    2     4    14 ... 34140 34142 34146]
FOLD:  3 TRAIN: [    0     1     2 ... 34148 34149 34150] TEST: [    3     6     8 ... 34133 34143 34151]
FOLD:  4 TRAIN: [    2     3     4 ... 34149 34150 34151] TEST: [    0     1    10 ... 34122 34124 34141]


In [53]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
data_encoded_bow = vectorizer.fit_transform(data["text"])
data_test_encoded_bow = vectorizer.transform(data_test["text"])

In [54]:
fit_predict_save(LogisticRegression(), "LogisticRegression_BOW", data_encoded_bow, data["labels"], data_test_encoded_bow)


# This model supports only non-negative values, so we need to shift the data to be non-negative
min_val = np.min([np.min(data_encoded_bow), np.min(data_test_encoded_bow)])
data_positive_encoded = data_encoded_bow - min_val
data_test_positive_encoded = data_test_encoded_bow - min_val

fit_predict_save(MultinomialNB(), "MultinomialNB_BOW", data_positive_encoded, data["labels"], data_test_positive_encoded)
fit_predict_save(RandomForestClassifier(), "RandomForestClassifier_BOW", data_encoded_bow, data["labels"], data_test_encoded_bow)

FOLD:  0 TRAIN: [    0     1     2 ... 34147 34149 34151] TEST: [    5     7    12 ... 34145 34148 34150]
FOLD:  1 TRAIN: [    0     1     2 ... 34148 34150 34151] TEST: [    9    15    25 ... 34144 34147 34149]
FOLD:  2 TRAIN: [    0     1     3 ... 34149 34150 34151] TEST: [    2     4    14 ... 34140 34142 34146]
FOLD:  3 TRAIN: [    0     1     2 ... 34148 34149 34150] TEST: [    3     6     8 ... 34133 34143 34151]
FOLD:  4 TRAIN: [    2     3     4 ... 34149 34150 34151] TEST: [    0     1    10 ... 34122 34124 34141]
FOLD:  0 TRAIN: [    0     1     2 ... 34147 34149 34151] TEST: [    5     7    12 ... 34145 34148 34150]
FOLD:  1 TRAIN: [    0     1     2 ... 34148 34150 34151] TEST: [    9    15    25 ... 34144 34147 34149]
FOLD:  2 TRAIN: [    0     1     3 ... 34149 34150 34151] TEST: [    2     4    14 ... 34140 34142 34146]
FOLD:  3 TRAIN: [    0     1     2 ... 34148 34149 34150] TEST: [    3     6     8 ... 34133 34143 34151]
FOLD:  4 TRAIN: [    2     3     4 ... 34149 3

In [55]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
data_encoded = vectorizer.fit_transform(data["text"])
data_test_encoded = vectorizer.transform(data_test["text"])

In [ ]:
fit_predict_save(LogisticRegression(), "LogisticRegression_TFIDF", data_encoded, data["labels"], data_test_encoded)


# This model supports only non-negative values, so we need to shift the data to be non-negative
min_val = np.min([np.min(data_encoded), np.min(data_test_encoded)])
data_positive_encoded = data_encoded - min_val
data_test_positive_encoded = data_test_encoded - min_val

fit_predict_save(MultinomialNB(), "MultinomialNB_TFIDF", data_positive_encoded, data["labels"], data_test_positive_encoded)
fit_predict_save(RandomForestClassifier(), "RandomForestClassifier_TFIDF", data_encoded, data["labels"], data_test_encoded)

FOLD:  0 TRAIN: [    0     1     2 ... 34147 34149 34151] TEST: [    5     7    12 ... 34145 34148 34150]
FOLD:  1 TRAIN: [    0     1     2 ... 34148 34150 34151] TEST: [    9    15    25 ... 34144 34147 34149]
FOLD:  2 TRAIN: [    0     1     3 ... 34149 34150 34151] TEST: [    2     4    14 ... 34140 34142 34146]
FOLD:  3 TRAIN: [    0     1     2 ... 34148 34149 34150] TEST: [    3     6     8 ... 34133 34143 34151]
FOLD:  4 TRAIN: [    2     3     4 ... 34149 34150 34151] TEST: [    0     1    10 ... 34122 34124 34141]
FOLD:  0 TRAIN: [    0     1     2 ... 34147 34149 34151] TEST: [    5     7    12 ... 34145 34148 34150]
FOLD:  1 TRAIN: [    0     1     2 ... 34148 34150 34151] TEST: [    9    15    25 ... 34144 34147 34149]
FOLD:  2 TRAIN: [    0     1     3 ... 34149 34150 34151] TEST: [    2     4    14 ... 34140 34142 34146]
FOLD:  3 TRAIN: [    0     1     2 ... 34148 34149 34150] TEST: [    3     6     8 ... 34133 34143 34151]
FOLD:  4 TRAIN: [    2     3     4 ... 34149 3

: 

In [11]:
# ============================================================
# SECTION 1: More classifiers on existing MiniLM embeddings
# (data_encoded / data_test_encoded already computed above)
# ============================================================

from sklearn.svm import LinearSVC
from sklearn.linear_model import SGDClassifier, RidgeClassifier
from sklearn.calibration import CalibratedClassifierCV

# LogisticRegression with different C values (regularization strength)
for C in [0.01, 0.1, 10]:
    fit_predict_save(
        LogisticRegression(C=C, max_iter=1000),
        f"LR_MiniLM_C{C}",
        data_encoded, data["labels"], data_test_encoded
    )

# LinearSVC — doesn't have predict_proba natively, so wrap with CalibratedClassifierCV
fit_predict_save(
    CalibratedClassifierCV(LinearSVC()),
    "LinearSVC_MiniLM",
    data_encoded, data["labels"], data_test_encoded
)

# SGDClassifier with log_loss (equivalent to logistic regression but different solver)
fit_predict_save(
    SGDClassifier(loss="log_loss", random_state=42),
    "SGD_log_MiniLM",
    data_encoded, data["labels"], data_test_encoded
)

# RidgeClassifier — also needs calibration for probabilities
fit_predict_save(
    CalibratedClassifierCV(RidgeClassifier()),
    "Ridge_MiniLM",
    data_encoded, data["labels"], data_test_encoded
)

FOLD:  0 TRAIN: [    0     1     2 ... 34147 34149 34151] TEST: [    5     7    12 ... 34145 34148 34150]
FOLD:  1 TRAIN: [    0     1     2 ... 34148 34150 34151] TEST: [    9    15    25 ... 34144 34147 34149]
FOLD:  2 TRAIN: [    0     1     3 ... 34149 34150 34151] TEST: [    2     4    14 ... 34140 34142 34146]
FOLD:  3 TRAIN: [    0     1     2 ... 34148 34149 34150] TEST: [    3     6     8 ... 34133 34143 34151]
FOLD:  4 TRAIN: [    2     3     4 ... 34149 34150 34151] TEST: [    0     1    10 ... 34122 34124 34141]
FOLD:  0 TRAIN: [    0     1     2 ... 34147 34149 34151] TEST: [    5     7    12 ... 34145 34148 34150]
FOLD:  1 TRAIN: [    0     1     2 ... 34148 34150 34151] TEST: [    9    15    25 ... 34144 34147 34149]
FOLD:  2 TRAIN: [    0     1     3 ... 34149 34150 34151] TEST: [    2     4    14 ... 34140 34142 34146]
FOLD:  3 TRAIN: [    0     1     2 ... 34148 34149 34150] TEST: [    3     6     8 ... 34133 34143 34151]
FOLD:  4 TRAIN: [    2     3     4 ... 34149 3

In [13]:
# ============================================================
# SECTION 2: TF-IDF variants
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

# Unigrams with large vocab
vec_tfidf_large = TfidfVectorizer(max_features=100_000)
X_tfidf_large = vec_tfidf_large.fit_transform(data["text"])
X_test_tfidf_large = vec_tfidf_large.transform(data_test["text"])

fit_predict_save(LogisticRegression(max_iter=1000), "LR_TFIDF_100k",
                 X_tfidf_large, data["labels"], X_test_tfidf_large)
fit_predict_save(CalibratedClassifierCV(LinearSVC()), "LinearSVC_TFIDF_100k",
                 X_tfidf_large, data["labels"], X_test_tfidf_large)

# Bigrams
vec_tfidf_bigram = TfidfVectorizer(ngram_range=(1, 2), max_features=100_000)
X_tfidf_bigram = vec_tfidf_bigram.fit_transform(data["text"])
X_test_tfidf_bigram = vec_tfidf_bigram.transform(data_test["text"])

fit_predict_save(LogisticRegression(max_iter=1000), "LR_TFIDF_bigram",
                 X_tfidf_bigram, data["labels"], X_test_tfidf_bigram)
fit_predict_save(CalibratedClassifierCV(LinearSVC()), "LinearSVC_TFIDF_bigram",
                 X_tfidf_bigram, data["labels"], X_test_tfidf_bigram)

# Character n-grams — captures spelling/punctuation patterns typical of fake news
vec_tfidf_char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=100_000)
X_tfidf_char = vec_tfidf_char.fit_transform(data["text"])
X_test_tfidf_char = vec_tfidf_char.transform(data_test["text"])

fit_predict_save(LogisticRegression(max_iter=1000), "LR_TFIDF_char",
                 X_tfidf_char, data["labels"], X_test_tfidf_char)
fit_predict_save(CalibratedClassifierCV(LinearSVC()), "LinearSVC_TFIDF_char",
                 X_tfidf_char, data["labels"], X_test_tfidf_char)

# NaiveBayes on char n-grams (needs non-negative — TF-IDF is always >= 0 so no shift needed)
fit_predict_save(MultinomialNB(), "NB_TFIDF_char",
                 X_tfidf_char, data["labels"], X_test_tfidf_char)

FOLD:  0 TRAIN: [    0     1     2 ... 34147 34149 34151] TEST: [    5     7    12 ... 34145 34148 34150]
FOLD:  1 TRAIN: [    0     1     2 ... 34148 34150 34151] TEST: [    9    15    25 ... 34144 34147 34149]
FOLD:  2 TRAIN: [    0     1     3 ... 34149 34150 34151] TEST: [    2     4    14 ... 34140 34142 34146]
FOLD:  3 TRAIN: [    0     1     2 ... 34148 34149 34150] TEST: [    3     6     8 ... 34133 34143 34151]
FOLD:  4 TRAIN: [    2     3     4 ... 34149 34150 34151] TEST: [    0     1    10 ... 34122 34124 34141]
FOLD:  0 TRAIN: [    0     1     2 ... 34147 34149 34151] TEST: [    5     7    12 ... 34145 34148 34150]
FOLD:  1 TRAIN: [    0     1     2 ... 34148 34150 34151] TEST: [    9    15    25 ... 34144 34147 34149]
FOLD:  2 TRAIN: [    0     1     3 ... 34149 34150 34151] TEST: [    2     4    14 ... 34140 34142 34146]
FOLD:  3 TRAIN: [    0     1     2 ... 34148 34149 34150] TEST: [    3     6     8 ... 34133 34143 34151]
FOLD:  4 TRAIN: [    2     3     4 ... 34149 3

In [14]:
# ============================================================
# SECTION 3: Heavier sentence transformer (mpnet)
# ============================================================

from sentence_transformers import SentenceTransformer

model_mpnet = SentenceTransformer("all-mpnet-base-v2")
X_mpnet = model_mpnet.encode(data["text"].values, show_progress_bar=True)
X_test_mpnet = model_mpnet.encode(data_test["text"].values, show_progress_bar=True)

fit_predict_save(LogisticRegression(max_iter=1000), "LR_mpnet",
                 X_mpnet, data["labels"], X_test_mpnet)
fit_predict_save(CalibratedClassifierCV(LinearSVC()), "LinearSVC_mpnet",
                 X_mpnet, data["labels"], X_test_mpnet)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8865.38it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 312/312 [00:19<00:00, 15.77it/s]


FOLD:  0 TRAIN: [    0     1     2 ... 34147 34149 34151] TEST: [    5     7    12 ... 34145 34148 34150]
FOLD:  1 TRAIN: [    0     1     2 ... 34148 34150 34151] TEST: [    9    15    25 ... 34144 34147 34149]
FOLD:  2 TRAIN: [    0     1     3 ... 34149 34150 34151] TEST: [    2     4    14 ... 34140 34142 34146]
FOLD:  3 TRAIN: [    0     1     2 ... 34148 34149 34150] TEST: [    3     6     8 ... 34133 34143 34151]
FOLD:  4 TRAIN: [    2     3     4 ... 34149 34150 34151] TEST: [    0     1    10 ... 34122 34124 34141]
FOLD:  0 TRAIN: [    0     1     2 ... 34147 34149 34151] TEST: [    5     7    12 ... 34145 34148 34150]
FOLD:  1 TRAIN: [    0     1     2 ... 34148 34150 34151] TEST: [    9    15    25 ... 34144 34147 34149]
FOLD:  2 TRAIN: [    0     1     3 ... 34149 34150 34151] TEST: [    2     4    14 ... 34140 34142 34146]
FOLD:  3 TRAIN: [    0     1     2 ... 34148 34149 34150] TEST: [    3     6     8 ... 34133 34143 34151]
FOLD:  4 TRAIN: [    2     3     4 ... 34149 3

In [16]:
# ============================================================
# SECTION 4: MLP on MiniLM embeddings (PyTorch + MPS)
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = (
    torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)
print(f"Using device: {device}")


class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))  # single output → sigmoid → probability
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)


def train_mlp(X_train, y_train, X_val, hidden_dims, dropout, lr, epochs, batch_size):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train.values, dtype=torch.float32).to(device)
    X_v  = torch.tensor(X_val,   dtype=torch.float32).to(device)

    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)

    model = MLP(X_train.shape[1], hidden_dims, dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    model.train()
    for epoch in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        logits = model(X_v).cpu().numpy()
    probs = 1 / (1 + np.exp(-logits))  # sigmoid
    return model, probs


def fit_predict_save_mlp(MODEL_NAME, X, y, X_test,
                          hidden_dims, dropout=0.3, lr=1e-3,
                          epochs=20, batch_size=256):
    oof  = np.zeros(len(X))
    pred = np.zeros(len(X_test))

    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

    for i, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        print(f"FOLD: {i}")
        X_train, X_val = X[train_idx], X[val_idx]
        y_train = y.iloc[train_idx] if hasattr(y, 'iloc') else y[train_idx]

        model, val_probs = train_mlp(X_train, y_train if hasattr(y_train, 'values') else
                                     pd.Series(y_train), X_val,
                                     hidden_dims, dropout, lr, epochs, batch_size)

        oof[val_idx] = val_probs

        model.eval()
        with torch.no_grad():
            logits = model(X_test_t).cpu().numpy()
        pred += 1 / (1 + np.exp(-logits))

    pred /= n_folds
    pd.DataFrame({MODEL_NAME: pred}).to_csv(f"results/{MODEL_NAME}.csv", index=False)
    pd.DataFrame({MODEL_NAME + "_oof": oof}).to_csv(f"results/{MODEL_NAME}_oof.csv", index=False)
    print(f"Saved {MODEL_NAME}")


# --- Small MLP: 2 hidden layers ---
fit_predict_save_mlp(
    "MLP_small_MiniLM", data_encoded, data["labels"], data_test_encoded,
    hidden_dims=[256, 64], dropout=0.3, lr=1e-3, epochs=20
)

# --- Medium MLP: 3 hidden layers ---
fit_predict_save_mlp(
    "MLP_medium_MiniLM", data_encoded, data["labels"], data_test_encoded,
    hidden_dims=[512, 256, 64], dropout=0.3, lr=1e-3, epochs=20
)

# --- Same architectures on mpnet embeddings (if you ran Section 3) ---
fit_predict_save_mlp(
    "MLP_small_mpnet", X_mpnet, data["labels"], X_test_mpnet,
    hidden_dims=[256, 64], dropout=0.3, lr=1e-3, epochs=20
)
fit_predict_save_mlp(
    "MLP_medium_mpnet", X_mpnet, data["labels"], X_test_mpnet,
    hidden_dims=[512, 256, 64], dropout=0.3, lr=1e-3, epochs=20
)

Using device: mps
FOLD: 0
FOLD: 1
FOLD: 2
FOLD: 3
FOLD: 4
Saved MLP_small_MiniLM
FOLD: 0
FOLD: 1
FOLD: 2
FOLD: 3
FOLD: 4
Saved MLP_medium_MiniLM
FOLD: 0
FOLD: 1
FOLD: 2
FOLD: 3
FOLD: 4
Saved MLP_small_mpnet
FOLD: 0
FOLD: 1
FOLD: 2
FOLD: 3
FOLD: 4
Saved MLP_medium_mpnet


In [18]:
# ============================================================
# SECTION 5: CNN and LSTM on word-level token sequences
# ============================================================

from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from collections import Counter
import re

# ── Tokenizer & Vocabulary ───────────────────────────────────

def simple_tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())

def build_vocab(texts, max_vocab=30_000):
    counter = Counter()
    for t in texts:
        counter.update(simple_tokenize(t))
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for word, _ in counter.most_common(max_vocab - 2):
        vocab[word] = len(vocab)
    return vocab

def encode_texts(texts, vocab, max_len=50):
    """Returns (N, max_len) int array, padded/truncated."""
    unk = vocab["<UNK>"]
    pad = vocab["<PAD>"]
    result = np.full((len(texts), max_len), pad, dtype=np.int64)
    lengths = np.zeros(len(texts), dtype=np.int64)
    for i, text in enumerate(texts):
        tokens = simple_tokenize(text)[:max_len]
        for j, tok in enumerate(tokens):
            result[i, j] = vocab.get(tok, unk)
        lengths[i] = max(len(tokens), 1)
    return result, lengths

# Build vocab on training text only (no leakage)
vocab = build_vocab(data["text"].values)
VOCAB_SIZE = len(vocab)
MAX_LEN = 50
EMBED_DIM = 64

print(f"Vocabulary size: {VOCAB_SIZE}")

X_seq, X_lengths         = encode_texts(data["text"].values,      vocab, MAX_LEN)
X_test_seq, X_test_lens  = encode_texts(data_test["text"].values, vocab, MAX_LEN)


# ── CNN Model ────────────────────────────────────────────────

class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_filters=128, kernel_sizes=(2, 3, 4), dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, k) for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), 1)

    def forward(self, x):
        # x: (batch, seq_len)
        emb = self.embedding(x).permute(0, 2, 1)   # (batch, embed_dim, seq_len)
        pooled = [torch.relu(conv(emb)).max(dim=2).values for conv in self.convs]
        out = torch.cat(pooled, dim=1)              # (batch, num_filters * len(kernel_sizes))
        return self.fc(self.dropout(out)).squeeze(1)


# ── LSTM Model ───────────────────────────────────────────────

class TextLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0,
                            bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)  # *2 for bidirectional

    def forward(self, x, lengths):
        emb = self.embedding(x)                 # (batch, seq_len, embed_dim)
        packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        # hidden: (num_layers*2, batch, hidden_dim) — take last layer, both directions
        hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)  # (batch, hidden_dim*2)
        return self.fc(self.dropout(hidden)).squeeze(1)


# ── Generic training loop for sequence models ───────────────

def train_seq_model(model, X_train_seq, y_train, X_val_seq,
                    lengths_train=None, lengths_val=None,
                    lr=1e-3, epochs=10, batch_size=256):

    X_tr = torch.tensor(X_train_seq, dtype=torch.long).to(device)
    y_tr = torch.tensor(y_train if isinstance(y_train, np.ndarray)
                         else y_train.values, dtype=torch.float32).to(device)
    X_v  = torch.tensor(X_val_seq, dtype=torch.long).to(device)

    if lengths_train is not None:
        l_tr = torch.tensor(lengths_train, dtype=torch.long)
        l_v  = torch.tensor(lengths_val,   dtype=torch.long)
        dataset = TensorDataset(X_tr, y_tr, l_tr.to(device))
        use_lengths = True
    else:
        dataset = TensorDataset(X_tr, y_tr)
        use_lengths = False

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    model.train()
    for epoch in range(epochs):
        for batch in loader:
            optimizer.zero_grad()
            if use_lengths:
                xb, yb, lb = batch
                logits = model(xb, lb)
            else:
                xb, yb = batch
                logits = model(xb)
            criterion(logits, yb).backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        if use_lengths:
            logits = model(X_v, l_v.to(device)).cpu().numpy()
        else:
            logits = model(X_v).cpu().numpy()

    return 1 / (1 + np.exp(-logits))  # sigmoid → probabilities


# ── fit_predict_save for sequence models ────────────────────

def fit_predict_save_seq(MODEL_NAME, model_fn, X_seq, y, X_test_seq,
                          lengths=None, test_lengths=None,
                          lr=1e-3, epochs=10, batch_size=256):
    """
    model_fn: callable that returns a fresh model instance each fold
    lengths / test_lengths: required for LSTM, None for CNN
    """
    oof  = np.zeros(len(X_seq))
    pred = np.zeros(len(X_test_seq))

    X_test_t = torch.tensor(X_test_seq, dtype=torch.long).to(device)

    for i, (train_idx, val_idx) in enumerate(kf.split(X_seq, y)):
        print(f"FOLD: {i}")
        X_tr, X_v   = X_seq[train_idx], X_seq[val_idx]
        y_tr        = y.iloc[train_idx] if hasattr(y, 'iloc') else y[train_idx]
        l_tr = lengths[train_idx] if lengths is not None else None
        l_v  = lengths[val_idx]   if lengths is not None else None

        model = model_fn().to(device)
        val_probs = train_seq_model(model, X_tr, y_tr, X_v,
                                    l_tr, l_v, lr, epochs, batch_size)
        oof[val_idx] = val_probs

        model.eval()
        with torch.no_grad():
            if test_lengths is not None:
                logits = model(X_test_t, torch.tensor(test_lengths).to(device)).cpu().numpy()
            else:
                logits = model(X_test_t).cpu().numpy()
        pred += 1 / (1 + np.exp(-logits))

    pred /= n_folds
    pd.DataFrame({MODEL_NAME: pred}).to_csv(f"results/{MODEL_NAME}.csv", index=False)
    pd.DataFrame({MODEL_NAME + "_oof": oof}).to_csv(f"results/{MODEL_NAME}_oof.csv", index=False)
    print(f"Saved {MODEL_NAME}")


# ── Run CNN models ───────────────────────────────────────────

# Small CNN
fit_predict_save_seq(
    "CNN_small",
    lambda: TextCNN(VOCAB_SIZE, EMBED_DIM, num_filters=128, kernel_sizes=(2, 3, 4)),
    X_seq, data["labels"], X_test_seq,
    epochs=10
)

# Larger CNN with more filter sizes
fit_predict_save_seq(
    "CNN_large",
    lambda: TextCNN(VOCAB_SIZE, EMBED_DIM, num_filters=256, kernel_sizes=(2, 3, 4, 5)),
    X_seq, data["labels"], X_test_seq,
    epochs=10
)


# ── Run LSTM models ──────────────────────────────────────────

# Small bidirectional LSTM
fit_predict_save_seq(
    "LSTM_small",
    lambda: TextLSTM(VOCAB_SIZE, EMBED_DIM, hidden_dim=128, num_layers=1),
    X_seq, data["labels"], X_test_seq,
    lengths=X_lengths, test_lengths=X_test_lens,
    epochs=10
)

# Deeper bidirectional LSTM
fit_predict_save_seq(
    "LSTM_deep",
    lambda: TextLSTM(VOCAB_SIZE, EMBED_DIM, hidden_dim=256, num_layers=2),
    X_seq, data["labels"], X_test_seq,
    lengths=X_lengths, test_lengths=X_test_lens,
    epochs=10
)

Vocabulary size: 18716
FOLD: 0
FOLD: 1
FOLD: 2
FOLD: 3
FOLD: 4
Saved CNN_small
FOLD: 0
FOLD: 1
FOLD: 2
FOLD: 3
FOLD: 4
Saved CNN_large
FOLD: 0
FOLD: 1
FOLD: 2
FOLD: 3
FOLD: 4
Saved LSTM_small
FOLD: 0
FOLD: 1
FOLD: 2
FOLD: 3
FOLD: 4
Saved LSTM_deep


In [22]:
# ============================================================
# SECTION 6: Fine-tuned DistilBERT
# ============================================================

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim.lr_scheduler import LinearLR

BERT_MODEL_NAME = "distilbert-base-uncased"
BERT_MAX_LEN = 64   # headlines are short, 64 is enough and much faster than 512
BERT_EPOCHS = 6
BERT_LR = 2e-5
BERT_BATCH_SIZE = 32

tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)


def tokenize_texts(texts, max_len=BERT_MAX_LEN):
    return tokenizer(
        list(texts),
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )


def train_distilbert(train_encodings, y_train, val_encodings,
                     epochs=BERT_EPOCHS, lr=BERT_LR, batch_size=BERT_BATCH_SIZE):

    # Move to device
    train_input_ids      = train_encodings["input_ids"].to(device)
    train_attention_mask = train_encodings["attention_mask"].to(device)
    y_tr = torch.tensor(y_train if isinstance(y_train, np.ndarray)
                         else y_train.values, dtype=torch.long).to(device)

    val_input_ids        = val_encodings["input_ids"].to(device)
    val_attention_mask   = val_encodings["attention_mask"].to(device)

    dataset = TensorDataset(train_input_ids, train_attention_mask, y_tr)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        BERT_MODEL_NAME, num_labels=2
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(loader) * epochs
    scheduler = LinearLR(optimizer, start_factor=1.0, end_factor=0.0, total_iters=total_steps)

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for input_ids, attention_mask, labels in loader:
            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            outputs.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # prevents exploding gradients
            optimizer.step()
            scheduler.step()
            total_loss += outputs.loss.item()
        print(f"  Epoch {epoch+1}/{epochs} — loss: {total_loss/len(loader):.4f}")

    # Validation predictions
    model.eval()
    all_probs = []
    with torch.no_grad():
        for start in range(0, val_input_ids.shape[0], batch_size):
            ids   = val_input_ids[start:start+batch_size]
            mask  = val_attention_mask[start:start+batch_size]
            logits = model(input_ids=ids, attention_mask=mask).logits
            probs  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)

    return model, np.array(all_probs)


def fit_predict_save_bert(MODEL_NAME, texts_train, y, texts_test,
                           epochs=BERT_EPOCHS, lr=BERT_LR, batch_size=BERT_BATCH_SIZE):

    oof  = np.zeros(len(texts_train))
    pred = np.zeros(len(texts_test))

    # Tokenize test set once — same for every fold
    test_encodings = tokenize_texts(texts_test)
    test_input_ids      = test_encodings["input_ids"].to(device)
    test_attention_mask = test_encodings["attention_mask"].to(device)

    for i, (train_idx, val_idx) in enumerate(kf.split(np.zeros(len(texts_train)), y)):
        print(f"FOLD: {i}")

        train_enc = tokenize_texts(texts_train[train_idx])
        val_enc   = tokenize_texts(texts_train[val_idx])
        y_tr = y.iloc[train_idx] if hasattr(y, 'iloc') else y[train_idx]

        model, val_probs = train_distilbert(train_enc, y_tr, val_enc, epochs, lr, batch_size)
        oof[val_idx] = val_probs

        # Test predictions
        model.eval()
        test_probs = []
        with torch.no_grad():
            for start in range(0, test_input_ids.shape[0], batch_size):
                ids  = test_input_ids[start:start+batch_size]
                mask = test_attention_mask[start:start+batch_size]
                logits = model(input_ids=ids, attention_mask=mask).logits
                probs  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
                test_probs.extend(probs)
        pred += np.array(test_probs)

        # Free VRAM/MPS memory between folds
        del model
        if device.type == "mps":
            torch.mps.empty_cache()
        elif device.type == "cuda":
            torch.cuda.empty_cache()

    pred /= n_folds
    pd.DataFrame({MODEL_NAME: pred}).to_csv(f"results/{MODEL_NAME}.csv", index=False)
    pd.DataFrame({MODEL_NAME + "_oof": oof}).to_csv(f"results/{MODEL_NAME}_oof.csv", index=False)
    print(f"Saved {MODEL_NAME}")


# ── Run ──────────────────────────────────────────────────────

texts_train = data["text"].values
texts_test  = data_test["text"].values

# DistilBERT default hyperparameters
fit_predict_save_bert(
    "DistilBERT_1",
    texts_train, data["labels"], texts_test
)

# Slightly more conservative LR — often gives a meaningfully different model
fit_predict_save_bert(
    "DistilBERT_2",
    texts_train, data["labels"], texts_test,
    lr=1e-5
)

FOLD: 0


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6717.34it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1/6 — loss: 0.1145
  Epoch 2/6 — loss: 0.0425
  Epoch 3/6 — loss: 0.0183
  Epoch 4/6 — loss: 0.0068
  Epoch 5/6 — loss: 0.0030
  Epoch 6/6 — loss: 0.0006
FOLD: 1


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5888.56it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1/6 — loss: 0.1163
  Epoch 2/6 — loss: 0.0431
  Epoch 3/6 — loss: 0.0190
  Epoch 4/6 — loss: 0.0083
  Epoch 5/6 — loss: 0.0023
  Epoch 6/6 — loss: 0.0012
FOLD: 2


...

  Epoch 1/6 — loss: 0.1360
  Epoch 2/6 — loss: 0.0564
  Epoch 3/6 — loss: 0.0335
  Epoch 4/6 — loss: 0.0189
  Epoch 5/6 — loss: 0.0102
  Epoch 6/6 — loss: 0.0073
FOLD: 3


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4936.16it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1/6 — loss: 0.1360
  Epoch 2/6 — loss: 0.0585
  Epoch 3/6 — loss: 0.0351
  Epoch 4/6 — loss: 0.0189
  Epoch 5/6 — loss: 0.0106
  Epoch 6/6 — loss: 0.0077
FOLD: 4


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6371.51it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Epoch 1/6 — loss: 0.1351
  Epoch 2/6 — loss: 0.0577
  Epoch 3/6 — loss: 0.0321
  Epoch 4/6 — loss: 0.0183
  Epoch 5/6 — loss: 0.0132
  Epoch 6/6 — loss: 0.0077
Saved DistilBERT_2
